<a href="https://colab.research.google.com/github/demerchantsean-wq/Exercise-Dashboard/blob/main/Exercise_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# 1. Install the required packages (runs silently)
!pip install -q gpxpy osmnx PyWavelets scipy timezonefinder

import gpxpy
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm
from folium.features import ColorLine
from google.colab import drive, output
import os
import re
import json
import time
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import pywt
from scipy.signal import correlate, correlation_lags
import requests
import matplotlib.ticker as ticker
from timezonefinder import TimezoneFinder

# --- Backend Bridge for Persistent Plot States ---
global_trace_presets = {}
global_startup_trace_preset = 'Default'

def update_trace_presets_callback(*args, **kwargs):
    global global_trace_presets, global_startup_trace_preset
    try:
        if len(args) > 0:
            payload_str = args[0]
        else:
            return

        data = json.loads(payload_str)
        action = data.get("action")
        name = data.get("name")

        settings = {}
        if os.path.exists(SETTINGS_FILE):
            try:
                with open(SETTINGS_FILE, 'r') as f:
                    settings = json.load(f)
            except: pass

        if action == 'save':
            states = data.get("states")
            if 'trace_presets' not in settings: settings['trace_presets'] = {}
            settings['trace_presets'][name] = states
            global_trace_presets[name] = states
        elif action == 'set_default':
            settings['startup_trace_preset'] = name
            global_startup_trace_preset = name

        with open(SETTINGS_FILE, 'w') as f:
            json.dump(settings, f)
    except Exception as e:
        print("Callback Error:", e)

output.register_callback('update_trace_presets', update_trace_presets_callback)

# --- Execution Profiler Timing ---
global_timing = {
    'tooltip_per_cell': 0.00005,
    'splom_per_cell': 0.00001,
    'map_per_pt': 0.00002
}

# Vectorized Haversine formula
def calculate_haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 3956 * c

# Signal Denoising
def denoise_signal(series, wavelet='sym4'):
    data = series.fillna(0).values
    if len(data) == 0: return series
    coeffs = pywt.wavedec(data, wavelet, mode='per')
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    if sigma == 0: sigma = 1e-9
    threshold = sigma * np.sqrt(2 * np.log(len(data)))
    coeffs[1:] = [pywt.threshold(c, value=threshold, mode='soft') for c in coeffs[1:]]
    return pd.Series(pywt.waverec(coeffs, wavelet, mode='per')[:len(data)])

# 2. Mount Drive
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# --- CONFIGURATION VARIABLES ---
BASE_FEATURES = [
    'hr', 'elevation_ft', 'slope_smoothed', 'elapsed_minutes',
    'mph_smoothed', 'hr_denoised', 'hr_aligned', 'slope_denoised'
]

DEFAULT_SELECTED_FEATURES = [
    'hr', 'elevation_ft', 'slope_smoothed', 'elapsed_minutes',
    'mph_smoothed', 'hr_aligned', 'slope_denoised'
]

SETTINGS_FILE = 'dashboard_settings.json'

# 3. Build the Folder Browser UI
start_path = '/content/drive/MyDrive/Biometrics and Environmental Data/EKG/Fourth Frontier'
if not os.path.exists(start_path): start_path = '/content'

path_label = widgets.Text(value=start_path, description='Current Dir:', disabled=True, layout=widgets.Layout(width='80%'))
dir_select = widgets.Select(options=[], description='Folders:', layout=widgets.Layout(width='80%', height='150px'))
btn_up = widgets.Button(description='⬆️ Up Level', button_style='warning')
btn_refresh = widgets.Button(description='🔄 Refresh Folder', button_style='success')
gpx_dropdown = widgets.Dropdown(options=[], description='GPX File:', disabled=True, layout=widgets.Layout(width='80%'))
hr_dropdown = widgets.Dropdown(options=['None'], description='Heart Rate:', disabled=True, layout=widgets.Layout(width='80%'))
rr_dropdown = widgets.Dropdown(options=['None'], description='R-R Interval:', disabled=True, layout=widgets.Layout(width='80%'))
summary_dropdown = widgets.Dropdown(options=['None'], description='Summary:', disabled=True, layout=widgets.Layout(width='80%'))
btn_load = widgets.Button(description='📂 Load Data & Extract Features', button_style='primary', disabled=True)

feature_box = widgets.VBox([])
btn_plot = widgets.Button(description='🗺️ Generate Maps & Interactive Plot', button_style='info', disabled=True)
progress_html = widgets.HTML(value="")
out = widgets.Output()

ui_container = widgets.VBox([])
btn_toggle_ui = widgets.Button(description='⚙️ Toggle Setup Controls', button_style='info')

def toggle_ui(b):
    if ui_container.layout.display == 'none':
        ui_container.layout.display = 'flex'
    else:
        ui_container.layout.display = 'none'

btn_toggle_ui.on_click(toggle_ui)

# Globals for passing data
global_df = pd.DataFrame()
global_route_points = []
global_bounds = []
global_summary_cols = []

checkbox_dict = {}
global_presets = {}
ui_preset_dropdown = None

feature_labels = {
    'hr': 'Heart Rate',
    'elevation_ft': 'Elevation',
    'slope_smoothed': 'Slope (%)',
    'elapsed_minutes': 'Elapsed Time',
    'mph_smoothed': 'Velocity (MPH)',
    'hr_denoised': 'Denoised Heart Rate',
    'hr_aligned': 'Aligned Heart Rate',
    'slope_denoised': 'Denoised Slope (%)'
}

def on_scan_clicked(b=None):
    with out:
        clear_output()
        folder = path_label.value
        try:
            gpx_files = [f for f in os.listdir(folder) if f.lower().endswith('.gpx')]
            csv_files = [f for f in os.listdir(folder) if f.lower().endswith('.csv')]

            if gpx_files:
                gpx_dropdown.options = sorted(gpx_files)
                gpx_dropdown.disabled = False
                btn_load.disabled = False
            else:
                gpx_dropdown.options = []
                gpx_dropdown.disabled = True
                btn_load.disabled = True

            csv_options = ['None'] + sorted(csv_files)
            is_csv_empty = len(csv_files) == 0

            hr_dropdown.options = csv_options
            hr_dropdown.disabled = is_csv_empty
            if '1sec_hr.csv' in csv_files: hr_dropdown.value = '1sec_hr.csv'

            rr_dropdown.options = csv_options
            rr_dropdown.disabled = is_csv_empty
            if 'rr_interval.csv' in csv_files: rr_dropdown.value = 'rr_interval.csv'

            summary_dropdown.options = csv_options
            summary_dropdown.disabled = is_csv_empty
            if 'summarydata.csv' in csv_files: summary_dropdown.value = 'summarydata.csv'

            print(f"Found {len(gpx_files)} GPX files and {len(csv_files)} CSV files.")
            print("Assign your files below and click 'Load Data & Extract Features'.")

            feature_box.children = []
            btn_plot.disabled = True
            progress_html.value = ""
        except Exception as e:
            print("Could not read directory contents.")

def on_dir_change(change):
    if change['new']:
        new_path = os.path.join(path_label.value, change['new'])
        update_browser(new_path)

def update_browser(path):
    try:
        path_label.value = path
        items = sorted([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)) and not d.startswith('.')])
        dir_select.unobserve(on_dir_change, names='value')
        dir_select.options = items
        dir_select.value = None
        dir_select.observe(on_dir_change, names='value')
        on_scan_clicked()
    except Exception as e:
        dir_select.unobserve(on_dir_change, names='value')
        dir_select.options = []
        dir_select.value = None
        dir_select.observe(on_dir_change, names='value')

def on_up_clicked(b):
    new_path = os.path.dirname(path_label.value)
    update_browser(new_path)

def load_csv(dropdown, name, folder):
    if dropdown.value != 'None':
        path = os.path.join(folder, dropdown.value)
        print(f"Loading {name} data from '{dropdown.value}'...")
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"❌ Error loading {name} CSV: {e}")
    return None

def process_data(b):
    global global_df, global_route_points, global_bounds, global_summary_cols
    global checkbox_dict, global_presets, global_timing, global_trace_presets, global_startup_trace_preset
    global ui_preset_dropdown

    with out:
        clear_output()
        folder = path_label.value

        hr_df = load_csv(hr_dropdown, "Heart Rate", folder)
        rr_df = load_csv(rr_dropdown, "R-R Interval", folder)
        summary_df = load_csv(summary_dropdown, "Summary Data", folder)

        gpx_path = os.path.join(folder, gpx_dropdown.value)
        print(f"Processing GPX route '{gpx_dropdown.value}'...\n")
        with open(gpx_path, 'r') as gpx_file:
            gpx = gpxpy.parse(gpx_file)

        route_data = []
        for track in gpx.tracks:
            for segment in track.segments:
                for point in segment.points:
                    route_data.append({
                        'lat': point.latitude,
                        'lon': point.longitude,
                        'elevation': point.elevation,
                        'time': point.time
                    })

        df = pd.DataFrame(route_data)
        if df.empty:
            print("Error: No track points found in this GPX file.")
            return

        df['elevation'] = df['elevation'].interpolate().bfill()
        df['elevation_ft'] = df['elevation'] * 3.28084

        summary_cols_added = []

        if not df['time'].isnull().all():
            df['time'] = pd.to_datetime(df['time'], utc=True)
            df = df.sort_values('time')

            if hr_df is not None and not hr_df.empty:
                if any('session_' in col for col in hr_df.columns):
                    def get_session_num(col_name):
                        match = re.search(r'session_(\d+)', col_name)
                        return int(match.group(1)) if match else -1

                    session_cols = sorted([c for c in hr_df.columns if 'session_' in c], key=get_session_num)
                    melted = hr_df[session_cols].melt().dropna(subset=['value'])
                    hr_values = melted['value'].astype(float).values
                    session_nums = [get_session_num(col) for col in melted['variable']]

                    gpx_start = df['time'].iloc[0]
                    hr_times = gpx_start + pd.to_timedelta(np.arange(len(hr_values)), unit='s')

                    hr_processed = pd.DataFrame({'merge_time': hr_times, 'hr': hr_values, 'session': session_nums})
                else:
                    time_col = hr_df.columns[0]
                    hr_col = hr_df.columns[1]
                    hr_processed = pd.DataFrame()
                    hr_processed['merge_time'] = pd.to_datetime(hr_df[time_col], utc=True, errors='coerce')
                    hr_processed['hr'] = hr_df[hr_col]
                    hr_processed['session'] = 1

                hr_processed = hr_processed.dropna(subset=['merge_time']).sort_values('merge_time')
                df = pd.merge_asof(
                    df, hr_processed[['merge_time', 'hr', 'session']],
                    left_on='time', right_on='merge_time', direction='nearest', tolerance=pd.Timedelta('15s')
                )
                df['hr'] = df['hr'].fillna(0)
                df['session'] = df['session'].fillna(-1).astype(int)
            else:
                df['hr'] = 0
                df['session'] = -1

            if summary_df is not None and not summary_df.empty:
                time_col_sum = next((col for col in summary_df.columns if 'time' in col.lower()), summary_df.columns[0])
                summary_df['merge_time_sum'] = pd.to_datetime(summary_df[time_col_sum], utc=True, errors='coerce')
                summary_df = summary_df.dropna(subset=['merge_time_sum']).sort_values('merge_time_sum')

                all_num_cols = summary_df.select_dtypes(include=[np.number]).columns.tolist()
                num_cols = [col for col in all_num_cols if col != time_col_sum]

                rename_dict = {col: f"Summary_{col}" for col in num_cols}
                summary_df_clean = summary_df[['merge_time_sum'] + num_cols].rename(columns=rename_dict)

                df = pd.merge_asof(
                    df, summary_df_clean,
                    left_on='time', right_on='merge_time_sum', direction='nearest', tolerance=pd.Timedelta('15s')
                )

                summary_cols_added = list(rename_dict.values())
                df[summary_cols_added] = df[summary_cols_added].fillna(0)

                for p_col in [c for c in summary_cols_added if 'power' in c.lower()]:
                    df[p_col + '_cumsum'] = df[p_col].cumsum()

            tf = TimezoneFinder()
            first_valid = df[['lat', 'lon']].first_valid_index()
            if first_valid is not None:
                local_tz = tf.timezone_at(lng=df.loc[first_valid, 'lon'], lat=df.loc[first_valid, 'lat']) or 'UTC'
            else:
                local_tz = 'UTC'

            df['time_local'] = df['time'].dt.tz_convert(local_tz)
            df['time_str'] = df['time_local'].dt.strftime('%I:%M:%S %p %Z')
            df['time_iso'] = df['time_local'].dt.strftime('%Y-%m-%d %H:%M:%S')

            df['elapsed_minutes'] = (df['time'] - df['time'].iloc[0]).dt.total_seconds() / 60.0
            df['elapsed_minutes'] = df['elapsed_minutes'].interpolate().bfill()

            df['dist_miles'] = calculate_haversine_distance(
                df['lon'].shift(1), df['lat'].shift(1), df['lon'], df['lat']
            )
            df['time_diff_hours'] = df['time'].diff().dt.total_seconds() / 3600.0

            df['mph'] = df['dist_miles'] / df['time_diff_hours']
            df['mph'] = df['mph'].replace([np.inf, -np.inf], np.nan).fillna(0)
            df['mph_smoothed'] = df['mph'].rolling(window=10, min_periods=1).mean()

            df['elev_diff_ft'] = df['elevation_ft'].diff()
            df['dist_ft'] = df['dist_miles'] * 5280.0

            df['slope_pct'] = (df['elev_diff_ft'] / df['dist_ft']) * 100
            df['slope_pct'] = df['slope_pct'].replace([np.inf, -np.inf], np.nan).fillna(0)
            df['slope_smoothed'] = df['slope_pct'].rolling(window=10, min_periods=1).mean()

            df['slope_denoised'] = denoise_signal(df['slope_pct'])
            df['hr_denoised'] = denoise_signal(df['hr'])

            norm_slope = df['slope_denoised'] - df['slope_denoised'].mean()
            norm_hr = df['hr_denoised'] - df['hr_denoised'].mean()

            if norm_hr.any() and norm_slope.any():
                corr = correlate(norm_hr, norm_slope, mode='full')
                lags = correlation_lags(len(norm_hr), len(norm_slope), mode='full')

                valid_indices = np.where((lags >= 0) & (lags <= 60))[0]
                if len(valid_indices) > 0:
                    optimal_lag = lags[valid_indices[np.argmax(corr[valid_indices])]]
                else:
                    optimal_lag = 0
            else:
                optimal_lag = 0

            df['hr_aligned'] = df['hr'].shift(-optimal_lag).ffill().bfill().fillna(0)
            hr_bins = [0, 119, 136, 153, 171, 300]
            df['metabolic_zone'] = pd.cut(df['hr_aligned'], labels=['Recovery', 'Fat Oxidation', 'Mixed Fuel', 'Lactate Threshold', 'Anaerobic'], bins=hr_bins)

        else:
            for col in ['elapsed_minutes', 'mph_smoothed', 'slope_smoothed', 'hr', 'hr_aligned', 'slope_denoised', 'hr_denoised']:
                df[col] = 0
            df['time_str'] = "N/A"
            df['time_iso'] = "1970-01-01 00:00:00"
            df['time_local'] = pd.to_datetime("1970-01-01 00:00:00").tz_localize('UTC')
            df['metabolic_zone'] = 'Recovery'
            df['session'] = -1

        global_df = df
        global_route_points = list(zip(df['lat'], df['lon']))
        global_bounds = [[df['lat'].min(), df['lon'].min()], [df['lat'].max(), df['lon'].max()]]
        global_summary_cols = summary_cols_added

        available_features = BASE_FEATURES + summary_cols_added
        default_summary_cols = [c for c in summary_cols_added if 'qos' not in c.lower()]
        fallback_features_list = DEFAULT_SELECTED_FEATURES + default_summary_cols

        # --- Persistent Settings Loader ---
        settings = {}
        if os.path.exists(SETTINGS_FILE):
            try:
                with open(SETTINGS_FILE, 'r') as f:
                    settings = json.load(f)
            except Exception as e: pass

        global_presets = settings.get('presets', {})
        if not global_presets:
            global_presets['Default'] = {'map': fallback_features_list.copy(), 'plot': fallback_features_list.copy()}

        startup_preset = settings.get('startup_preset', 'Default')
        if startup_preset not in global_presets:
            startup_preset = list(global_presets.keys())[0]

        active_preset = global_presets[startup_preset]
        saved_map_feats = active_preset.get('map', fallback_features_list.copy())
        saved_plot_feats = active_preset.get('plot', fallback_features_list.copy())
        global_timing.update(settings.get('timing_ema', {}))

        global_trace_presets.clear()
        if 'trace_presets' in settings:
            global_trace_presets.update(settings['trace_presets'])
        if not global_trace_presets:
            global_trace_presets['Default'] = {}
        global_startup_trace_preset = settings.get('startup_trace_preset', 'Default')

        checkbox_dict.clear()

        try: tz_abbr = df['time_local'].dropna().iloc[0].strftime('%Z')
        except: tz_abbr = 'Local Time'

        def get_display_name(feat_name):
            clean = feature_labels.get(feat_name, feat_name.replace("Summary_", ""))
            if 'epoch' in clean.lower() or 'epoch' in feat_name.lower(): return f"Local Time ({tz_abbr})"
            if feat_name in df.columns and pd.api.types.is_numeric_dtype(df[feat_name]):
                if ('time' in clean.lower() or 'time' in feat_name.lower()) and df[feat_name].max() > 1e8:
                    return f"Local Time ({tz_abbr})"
            return clean

        def create_feature_rows(feat_list):
            rows = [widgets.HBox([
                widgets.Label('Feature', layout=widgets.Layout(width='220px', font_weight='bold')),
                widgets.Label('Map', layout=widgets.Layout(width='45px', font_weight='bold')),
                widgets.Label('Plot', layout=widgets.Layout(width='45px', font_weight='bold'))
            ])]

            def make_observer(plot_box):
                def observer(change):
                    if change['new']: plot_box.value = True
                return observer

            for f in feat_list:
                clean_name = get_display_name(f)
                is_map_def = f in saved_map_feats
                is_plot_def = f in saved_plot_feats

                cb_m = widgets.Checkbox(value=is_map_def, indent=False, layout=widgets.Layout(width='45px'))
                cb_p = widgets.Checkbox(value=is_plot_def, indent=False, layout=widgets.Layout(width='45px'))
                cb_m.observe(make_observer(cb_p), names='value')

                checkbox_dict[f] = {'map': cb_m, 'plot': cb_p}
                rows.append(widgets.HBox([widgets.Label(clean_name, layout=widgets.Layout(width='220px')), cb_m, cb_p]))
            return widgets.VBox(rows, layout=widgets.Layout(margin='0 40px 0 0'))

        half = len(available_features) // 2 + len(available_features) % 2
        left_box = create_feature_rows(available_features[:half])
        right_box = create_feature_rows(available_features[half:])

        ui_grid = widgets.HBox([left_box, right_box], layout=widgets.Layout(width='100%', justify_content='flex-start'))

        ui_preset_dropdown = widgets.Dropdown(options=list(global_presets.keys()), value=startup_preset, description='Load Data:', layout=widgets.Layout(width='250px'))
        preset_name_input = widgets.Text(placeholder='New subset name...', description='Save Data:', layout=widgets.Layout(width='220px'))

        btn_save_preset = widgets.Button(description='💾 Save Data Features', button_style='success', layout=widgets.Layout(width='160px'))
        btn_set_startup = widgets.Button(description='⭐ Set Startup', button_style='warning', layout=widgets.Layout(width='110px'))
        btn_all = widgets.Button(description="✅ All", button_style='info', layout=widgets.Layout(width='60px'))
        btn_none = widgets.Button(description="❌ None", button_style='danger', layout=widgets.Layout(width='60px'))

        def apply_preset(change):
            if change['new'] and change['new'] in global_presets:
                preset = global_presets[change['new']]
                for f, cbs in checkbox_dict.items():
                    cbs['map'].value = (f in preset.get('map', []))
                    cbs['plot'].value = (f in preset.get('plot', []))

        ui_preset_dropdown.observe(apply_preset, names='value')

        def set_all(b):
            for cbs in checkbox_dict.values():
                cbs['map'].value = True; cbs['plot'].value = True

        def set_none(b):
            for cbs in checkbox_dict.values():
                cbs['map'].value = False; cbs['plot'].value = False

        def set_startup_preset(b):
            name = ui_preset_dropdown.value
            try:
                if os.path.exists(SETTINGS_FILE):
                    with open(SETTINGS_FILE, 'r') as f: cur_settings = json.load(f)
                else: cur_settings = {}
            except: cur_settings = {}
            cur_settings['startup_preset'] = name
            with open(SETTINGS_FILE, 'w') as f: json.dump(cur_settings, f)
            with out: print(f"✅ Data Preset '{name}' is now the default startup configuration.")

        def save_preset(b):
            name = preset_name_input.value.strip()
            if not name: name = ui_preset_dropdown.value

            current_map = [f for f, cbs in checkbox_dict.items() if cbs['map'].value]
            current_plot = [f for f, cbs in checkbox_dict.items() if cbs['plot'].value]
            global_presets[name] = {'map': current_map, 'plot': current_plot}

            try:
                if os.path.exists(SETTINGS_FILE):
                    with open(SETTINGS_FILE, 'r') as f: cur_settings = json.load(f)
                else: cur_settings = {}
            except: cur_settings = {}

            cur_settings['presets'] = global_presets
            cur_settings['timing_ema'] = global_timing
            with open(SETTINGS_FILE, 'w') as f: json.dump(cur_settings, f)

            ui_preset_dropdown.unobserve(apply_preset, names='value')
            ui_preset_dropdown.options = list(global_presets.keys())
            ui_preset_dropdown.value = name
            ui_preset_dropdown.observe(apply_preset, names='value')
            preset_name_input.value = ''
            with out: print(f"✅ Data Feature subset '{name}' saved successfully.")

        btn_all.on_click(set_all)
        btn_none.on_click(set_none)
        btn_set_startup.on_click(set_startup_preset)
        btn_save_preset.on_click(save_preset)

        preset_row = widgets.HBox([ui_preset_dropdown, btn_set_startup, preset_name_input, btn_save_preset, btn_all, btn_none], layout=widgets.Layout(margin='0 0 15px 0'))

        feature_box.children = [
            widgets.HTML("<h3>Select Data Subsets:</h3>"),
            preset_row,
            ui_grid
        ]

        btn_plot.disabled = False
        print("✅ Data successfully loaded and features extracted.")

def generate_visualizations(b):
    ui_container.layout.display = 'none'
    with out:
        clear_output()
        df = global_df
        total_rows = len(df)
        map_feats = [f for f, cbs in checkbox_dict.items() if cbs['map'].value]
        plot_feats = [f for f, cbs in checkbox_dict.items() if cbs['plot'].value]

        if not map_feats and not plot_feats:
            print("❌ No features selected. Please select at least one Map or Plot before generating.")
            return

        data_preset_name = feature_box.children[1].children[0].value

        # --- ML Progress Execution Estimator ---
        est_tt = global_timing['tooltip_per_cell'] * total_rows * len(set(map_feats + plot_feats))
        est_splom = global_timing['splom_per_cell'] * total_rows * len(plot_feats)**2 if plot_feats else 0
        est_map = global_timing['map_per_pt'] * total_rows * max(1, len(map_feats)) if map_feats else 0
        total_est = est_tt + est_splom + est_map + 0.01
        accum_est = 0

        last_ui_update_time = time.time()

        def throttled_update(phase_name, force=False):
            nonlocal last_ui_update_time
            if force or (time.time() - last_ui_update_time > 0.025):
                pct = min(99, int((accum_est / total_est) * 100))
                progress_html.value = f"<b style='color:#007bff; font-size:14px;'>⏳ {phase_name}... {pct}%</b>"
                last_ui_update_time = time.time()

        throttled_update("Initializing Runtime", force=True)

        if 'time_iso' not in df.columns:
            try: df['time_iso'] = df['time_local'].dt.strftime('%Y-%m-%d %H:%M:%S')
            except Exception: df['time_iso'] = "1970-01-01 00:00:00"

        try: tz_abbr = df['time_local'].dropna().iloc[0].strftime('%Z')
        except: tz_abbr = 'Local Time'

        def get_display_name(feat_name):
            clean = feature_labels.get(feat_name, feat_name.replace("Summary_", ""))
            if 'epoch' in clean.lower() or 'epoch' in feat_name.lower(): return f"Local Time ({tz_abbr})"
            if feat_name in df.columns and pd.api.types.is_numeric_dtype(df[feat_name]):
                if ('time' in clean.lower() or 'time' in feat_name.lower()) and df[feat_name].max() > 1e8:
                    return f"Local Time ({tz_abbr})"
            return clean

        tf = TimezoneFinder()
        first_valid = df[['lat', 'lon']].first_valid_index()
        if first_valid is not None: local_tz = tf.timezone_at(lng=df.loc[first_valid, 'lon'], lat=df.loc[first_valid, 'lat']) or 'UTC'
        else: local_tz = 'UTC'

        def format_epoch(val, tz_str, str_format):
            try:
                if val > 1e11: dt = pd.to_datetime(val, unit='ms', utc=True)
                else: dt = pd.to_datetime(val, unit='s', utc=True)
                return dt.tz_convert(tz_str).strftime(str_format)
            except: return str(val)

        folder = path_label.value
        cyclosm_tiles = 'https://{s}.tile-cyclosm.openstreetmap.fr/cyclosm/{z}/{x}/{y}.png'
        cyclosm_attr = 'CyclOSM | © OpenStreetMap contributors'
        timestamp_str = datetime.now().strftime('%Y%m%d%H%M%S')

        all_selected_feats = list(set(map_feats + plot_feats))
        colormaps = {}
        for feat in all_selected_feats:
            if feat == 'hr':
                if not df['hr'].eq(0).all():
                    valid_hr = df[df['hr'] > 0]['hr'].tolist()
                    min_hr, max_hr = min(valid_hr), max(valid_hr)
                    mid_green = 142.5
                    if max_hr <= mid_green: c_colors, c_index = ['yellow', 'green'], [min_hr, max_hr]
                    elif min_hr >= mid_green: c_colors, c_index = ['green', 'red'], [min_hr, max_hr]
                    else: c_colors, c_index = ['yellow', 'green', 'red'], [min_hr, mid_green, max_hr]
                    cmap = cm.LinearColormap(colors=c_colors, index=c_index, vmin=min_hr, vmax=max_hr)
                    cmap.caption = "Heart Rate"
                    colormaps['hr'] = cmap
                continue

            if feat in df.columns and not df[feat].isnull().all() and (df[feat] != 0).any():
                metrics = df[feat].tolist()
                min_m, max_m = min(metrics), max(metrics)
                if min_m == max_m: max_m = min_m + 1.0
                if 'slope' in feat.lower():
                    max_abs = max(abs(min_m), abs(max_m))
                    if max_abs == 0: max_abs = 1.0
                    cmap = cm.linear.viridis.scale(-max_abs, max_abs)
                else:
                    cmap = cm.linear.viridis.scale(min_m, max_m)

                cmap.caption = get_display_name(feat)
                colormaps[feat] = cmap

        # --- PRE-CALCULATE TOOLTIP HTML & PLAIN TEXT FOR CLIPBOARD ---
        t0 = time.time()

        tooltip_array = []
        tooltip_text_array = []
        for idx, row in df.iterrows():
            html = f"<div style='font-size: 12px; line-height: 1.4;'><b>Clock:</b> {row['time_str']}<br>"
            html += f"<b>GPS:</b> {row['lat']:.6f}, {row['lon']:.6f}<br>"
            text = f"Clock: {row['time_str']}\nGPS: {row['lat']:.6f}, {row['lon']:.6f}\n"

            if 'session' in df.columns and row['session'] != -1:
                html += f"<b>Session:</b> {int(row['session'])}<br>"
                text += f"Session: {int(row['session'])}\n"

            for f_tip in all_selected_feats:
                val = row[f_tip]
                c_name = get_display_name(f_tip)
                is_epoch = "Local Time" in c_name

                if f_tip == 'hr':
                    c_color = colormaps['hr'](val) if 'hr' in colormaps and val > 0 else '#808080'
                    hr_disp = f"{val:.0f} bpm" if val > 0 else "N/A"
                    html += f"<span style='color:{c_color}; font-size:14px;'>■</span> <b>HR:</b> {hr_disp} ({row.get('metabolic_zone', 'N/A')})<br>"
                    text += f"HR: {hr_disp} ({row.get('metabolic_zone', 'N/A')})\n"
                elif is_epoch:
                    c_color = colormaps[f_tip](val) if f_tip in colormaps else '#000'
                    val_str = format_epoch(val, local_tz, '%I:%M:%S %p')
                    html += f"<span style='color:{c_color}; font-size:14px;'>■</span> <b>{c_name}:</b> {val_str}<br>"
                    text += f"{c_name}: {val_str}\n"
                elif 'power' in f_tip.lower():
                    c_color = colormaps[f_tip](val) if f_tip in colormaps else '#000'
                    html += f"<span style='color:{c_color}; font-size:14px;'>■</span> <b>{c_name}:</b> {val:.2f} (Total: {row.get(f_tip + '_cumsum', 0):.2f})<br>"
                    text += f"{c_name}: {val:.2f} (Total: {row.get(f_tip + '_cumsum', 0):.2f})\n"
                else:
                    c_color = colormaps[f_tip](val) if f_tip in colormaps else '#000'
                    html += f"<span style='color:{c_color}; font-size:14px;'>■</span> <b>{c_name}:</b> {val:.1f}<br>"
                    text += f"{c_name}: {val:.1f}\n"

            html += "</div>"
            tooltip_array.append(html)
            tooltip_text_array.append(text.strip())

            if idx % max(1, total_rows // 20) == 0:
                accum_est += global_timing['tooltip_per_cell'] * max(1, total_rows // 20) * len(all_selected_feats)
                throttled_update("Processing Tooltips & Metadata")

        df['tooltip_html'] = tooltip_array
        df['tooltip_text'] = tooltip_text_array
        df['time_str_hover'] = df['time_str'].astype(str)

        t1 = time.time()
        global_timing['tooltip_per_cell'] = 0.7 * global_timing['tooltip_per_cell'] + 0.3 * ((t1 - t0) / max(1, total_rows * len(all_selected_feats)))

        ts_feats = [f for f in plot_feats if f in df.columns and f != 'elapsed_minutes']
        has_plots = len(ts_feats) > 0

        js_data = "[]"
        labels_json = "[]"
        feats_json = "[]"

        if has_plots:
            req_cols = list(set(['lat', 'lon', 'elapsed_minutes', 'time_iso', 'time_str_hover', 'tooltip_html', 'tooltip_text'] + ts_feats))
            js_data = df[req_cols].to_json(orient='records')
            labels_json = json.dumps([get_display_name(f) for f in ts_feats])
            feats_json = json.dumps(ts_feats)

        # --- PYTHON PAIRGRID EXPORT & COVARIANCE PREP ---
        has_splom = False
        cov_matrix_json = "[]"
        corr_matrix_json = "[]"
        valid_plot_cols_json = "[]"
        display_labels_map_json = "{}"

        if plot_feats:
            throttled_update("Generating Python Scatter Matrix", force=True)
            t0 = time.time()
            scatter_cols = [c for c in plot_feats if 'cumsum' not in c.lower() and c != 'elapsed_minutes'
                            and not ('epoch' in c.lower() or ('time' in c.lower() and df[c].max() > 1e8))]
            valid_plot_cols = [c for c in scatter_cols if c in df.columns and not df[c].isnull().all() and (df[c] != 0).any()]

            if valid_plot_cols:
                has_splom = True
                time_col = 'elapsed_minutes'
                df_plot = df.dropna(subset=valid_plot_cols + [time_col]).reset_index(drop=True)

                np.random.seed(42)
                df_jittered = df_plot.copy()
                for col in valid_plot_cols:
                    col_range = df_plot[col].max() - df_plot[col].min()
                    if col_range == 0: col_range = 1
                    noise = np.random.uniform(-0.015 * col_range, 0.015 * col_range, size=len(df_plot))
                    df_jittered[col] = df_plot[col] + noise

                scatter_cmap = plt.get_cmap('viridis')
                norm = plt.Normalize(df_jittered[time_col].min(), df_jittered[time_col].max())
                colors = scatter_cmap(norm(df_jittered[time_col]))
                num_time_bins = 15
                df_jittered['time_bin'] = pd.cut(df_jittered[time_col], bins=num_time_bins, labels=False)
                df_jittered['time_bin'] = df_jittered['time_bin'].fillna(0).astype(int)
                bin_centers = []
                for i in range(num_time_bins):
                    bin_data = df_jittered[df_jittered['time_bin'] == i][time_col]
                    if not bin_data.empty: bin_centers.append(bin_data.mean())
                    else: bin_centers.append(0)
                bin_colors = [scatter_cmap(norm(c)) for c in bin_centers]

                accum_est += est_splom * 0.1
                throttled_update("Generating Scatter Matrix... Setting up grid", force=True)

                plt.figure(figsize=(16, 16))
                g = sns.PairGrid(df_jittered, vars=valid_plot_cols)

                def hollow_scatter(x, y, **kwargs):
                    c = colors[x.index]
                    plt.scatter(x, y, facecolors='none', edgecolors=c, s=20, alpha=0.6, linewidths=1.0)

                accum_est += est_splom * 0.4
                throttled_update("Generating Scatter Matrix... Mapping off-diagonals", force=True)
                g.map_offdiag(hollow_scatter)

                def time_coded_4_way(x, **kwargs):
                    ax = plt.gca()
                    for spine in ['top', 'right', 'bottom', 'left']: ax.spines[spine].set_visible(False)
                    ax.set_yticks([])
                    ax.patch.set_alpha(0.0)
                    ax_bl = ax.inset_axes([0.0, 0.0, 0.48, 0.48])
                    ax_tl = ax.inset_axes([0.0, 0.52, 0.48, 0.48])
                    ax_br = ax.inset_axes([0.52, 0.0, 0.48, 0.48])
                    ax_tr = ax.inset_axes([0.52, 0.52, 0.48, 0.48])
                    for sub_ax in [ax_bl, ax_tl, ax_br, ax_tr]:
                        sub_ax.set_xticks([]); sub_ax.set_yticks([])
                    bins = np.histogram_bin_edges(x.dropna(), bins=20)
                    bins_data = []
                    for i in range(num_time_bins):
                        data_slice = x[df_jittered['time_bin'] == i].dropna()
                        if not data_slice.empty: bins_data.append(data_slice)
                        else: bins_data.append(pd.Series([], dtype=float))
                    ax_bl.hist(x.dropna(), bins=bins, color='slategray', edgecolor='none', alpha=0.9)
                    for i in range(num_time_bins):
                        if len(bins_data[i]) > 0: ax_tl.hist(bins_data[i], bins=bins, color=bin_colors[i], alpha=0.5, edgecolor='none')
                    for i in reversed(range(num_time_bins)):
                        if len(bins_data[i]) > 0: ax_br.hist(bins_data[i], bins=bins, color=bin_colors[i], alpha=0.5, edgecolor='none')
                    counts_list = []
                    for data_slice in bins_data:
                        counts, _ = np.histogram(data_slice, bins=bins)
                        counts_list.append(counts)
                    counts_matrix = np.array(counts_list)
                    total_counts = counts_matrix.sum(axis=0)
                    with np.errstate(divide='ignore', invalid='ignore'):
                        fractions_matrix = np.true_divide(counts_matrix, total_counts)
                        fractions_matrix[~np.isfinite(fractions_matrix)] = 0
                    bottom = np.zeros(len(bins)-1)
                    widths = np.diff(bins)
                    centers = bins[:-1] + widths/2
                    for i in range(num_time_bins):
                        ax_tr.bar(centers, fractions_matrix[i], width=widths, bottom=bottom, color=bin_colors[i], edgecolor='none', alpha=0.9, align='center')
                        bottom += fractions_matrix[i]
                    ax_tr.set_ylim(0, 1.05)

                accum_est += est_splom * 0.4
                throttled_update("Generating Scatter Matrix... Mapping diagonals", force=True)
                g.map_diag(time_coded_4_way)

                g.fig.subplots_adjust(right=0.91)
                cbar_ax = g.fig.add_axes([0.93, 0.15, 0.02, 0.7])
                sm = plt.cm.ScalarMappable(cmap=scatter_cmap, norm=norm)
                sm.set_array([])
                cbar = g.fig.colorbar(sm, cax=cbar_ax)
                try:
                    start_time_pd = df['time_local'].dropna().iloc[0]
                    def format_time(x, pos):
                        if pd.isnull(x): return ''
                        return (start_time_pd + pd.Timedelta(minutes=x)).strftime('%I:%M %p')
                    cbar.ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_time))
                except: pass

                cbar.set_label(f'Time ({tz_abbr})', rotation=270, labelpad=25)
                save_filename = f"{data_preset_name}_ScatterMatrix_{timestamp_str}.png"
                save_path = os.path.join(folder, save_filename)
                g.savefig(save_path, dpi=150, bbox_inches='tight')
                plt.close(g.fig)

                cov_matrix = df_plot[valid_plot_cols].cov()
                corr_matrix = df_plot[valid_plot_cols].corr()
                display_labels = [get_display_name(c) for c in valid_plot_cols]

                plt.figure(figsize=(5, 5))
                sns.heatmap(cov_matrix, annot=True, fmt=".2g", cmap='coolwarm', center=0,
                            xticklabels=False, yticklabels=False,
                            linewidths=0.5, cbar_kws={'label': 'Covariance'}, annot_kws={"size": 7})
                plt.title(f'Covariance Matrix: {data_preset_name}', fontsize=14, pad=15)
                plt.tight_layout()
                save_filename_cov = f"{data_preset_name}_CovarianceMatrix_{timestamp_str}.png"
                save_path_cov = os.path.join(folder, save_filename_cov)
                plt.savefig(save_path_cov, dpi=150, bbox_inches='tight')
                plt.close()
                print(f"\n✅ Rendered and saved static Scatter & Covariance matrices to directory.")

                cov_matrix_json = json.dumps(cov_matrix.fillna(0).values.tolist())
                corr_matrix_json = json.dumps(corr_matrix.fillna(0).values.tolist())
                valid_plot_cols_json = json.dumps(valid_plot_cols)
                display_labels_map = {c: get_display_name(c) for c in valid_plot_cols}
                display_labels_map_json = json.dumps(display_labels_map)

            t1 = time.time()
            global_timing['splom_per_cell'] = 0.7 * global_timing['splom_per_cell'] + 0.3 * ((t1 - t0) / max(1, total_rows * len(plot_feats)**2))

        # --- UNIFIED LINKED MAP & PLOT GENERATION ---
        if map_feats:
            throttled_update("Fetching Overpass Geo-Data", force=True)
            print(f"\n--- GENERATING MASTER INTERACTIVE MAP & PLOTS ---")

            map_feature_names = [get_display_name(f) for f in map_feats]
            map_feature_names_json = json.dumps(map_feature_names)
            first_layer_name = get_display_name(map_feats[0])

            f_feat = folium.Figure(width='100%', height=1000)
            m = folium.Map(tiles=None, width='100%', height='100%', zoom_control=False)
            folium.TileLayer(tiles=cyclosm_tiles, attr=cyclosm_attr, name='CyclOSM Map', control=False).add_to(m)

            north, south = df['lat'].max() + 0.005, df['lat'].min() - 0.005
            east, west = df['lon'].max() + 0.005, df['lon'].min() - 0.005

            overpass_endpoints = [
                "https://overpass-api.de/api/interpreter",
                "https://overpass.kumi.systems/api/interpreter",
                "https://lz4.overpass-api.de/api/interpreter"
            ]
            overpass_query = f"""[out:json][timeout:60];(node["amenity"="drinking_water"]({south},{west},{north},{east});way["shop"="bicycle"]({south},{west},{north},{east});node["shop"="bicycle"]({south},{west},{north},{east}););out center;"""

            headers = {'User-Agent': 'CyclingPerformanceDashboard/1.1 (Colab)'}
            poi_data = None
            for url in overpass_endpoints:
                try:
                    response = requests.post(url, data={'data': overpass_query}, headers=headers, timeout=65)
                    if response.status_code == 200:
                        poi_data = response.json()
                        break
                except Exception as e: pass

            if poi_data:
                for element in poi_data.get('elements', []):
                    if element['type'] == 'node': lat, lon = element['lat'], element['lon']
                    elif 'center' in element: lat, lon = element['center']['lat'], element['center']['lon']
                    else: continue
                    tags = element.get('tags', {})
                    is_water = tags.get('amenity') == 'drinking_water'
                    icon_color, icon_type = ('blue', 'tint') if is_water else ('orange', 'bicycle')
                    name = tags.get('name', 'Drinking Water' if is_water else 'Bike Shop')
                    folium.Marker(location=[lat, lon], tooltip=name, icon=folium.Icon(color=icon_color, icon=icon_type, prefix='fa')).add_to(m)

            layer_js_map = {}
            t0 = time.time()
            for i, feat in enumerate(map_feats):
                accum_est += global_timing['map_per_pt'] * total_rows
                throttled_update(f"Drawing Color Map Paths ({i+1}/{len(map_feats)})")
                clean_name = get_display_name(feat)
                fg = folium.FeatureGroup(name=clean_name, show=True, custom_name=clean_name)

                if feat == 'hr':
                    if 'hr' not in colormaps: ColorLine(positions=global_route_points, colors=df['hr'].tolist(), weight=6, opacity=0.8).add_to(fg)
                    else: ColorLine(positions=global_route_points, colors=df['hr'].tolist(), colormap=colormaps['hr'], weight=6).add_to(fg)
                else:
                    if feat not in colormaps: folium.PolyLine(global_route_points, color='gray', weight=6, opacity=0.8).add_to(fg)
                    else: ColorLine(positions=global_route_points, colors=df[feat].tolist(), colormap=colormaps[feat], weight=6).add_to(fg)
                fg.add_to(m)
                layer_js_map[clean_name] = fg.get_name()

            t1 = time.time()
            global_timing['map_per_pt'] = 0.7 * global_timing['map_per_pt'] + 0.3 * ((t1 - t0) / max(1, total_rows * len(map_feats)))

            folium.Marker(global_route_points[0], tooltip="Start", icon=folium.Icon(color='green', icon='play')).add_to(m)
            folium.Marker(global_route_points[-1], tooltip="End", icon=folium.Icon(color='red', icon='stop')).add_to(m)
            m.fit_bounds(global_bounds)
            m.add_to(f_feat)
            map_id = m.get_name()

            legends_html = "<div style='display: flex; flex-wrap: wrap; justify-content: space-evenly; gap: 15px;'>"
            legend_js_map = {}
            for feat in map_feats:
                clean_name = get_display_name(feat)
                if feat in colormaps:
                    svg_html = colormaps[feat]._repr_html_()
                    if "Local Time" in clean_name:
                        def replacer(match):
                            try:
                                val = float(match.group(1))
                                return f">{format_epoch(val, local_tz, '%I:%M %p')}<"
                            except:
                                return match.group(0)
                        svg_html = re.sub(r'>\s*([\d\.eE\+\-]+)\s*<', replacer, svg_html)
                    leg_id = f"leg_{abs(hash(clean_name))}"
                    legends_html += f"<div id='{leg_id}' style='display: none;'>{svg_html}</div>"
                    legend_js_map[clean_name] = leg_id
            legends_html += "</div>"

            legends_html_json = json.dumps(legends_html)
            legend_js_map_json = json.dumps(legend_js_map)
            plot_div_id = f"plot_master_{timestamp_str}"
            splom_div_id = f"splom_{timestamp_str}"
            cov_div_id = f"cov_{timestamp_str}"

            layer_js_map_json = json.dumps(layer_js_map)
            trace_presets_json = json.dumps(global_trace_presets)
            throttled_update("Assembling Interactive UI", force=True)

            custom_html = f"""<script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
            <style>
                body {{ margin: 0; padding: 0; overflow-x: hidden; box-sizing: border-box; }}
                * {{ box-sizing: border-box; }}
                #map_layer_select {{ max-width: 150px; }}
                :fullscreen #fs_container_{timestamp_str} {{ background-color: #f4f4f4; padding: 20px; overflow-y: auto; height: 100vh; }}
                :-webkit-full-screen #fs_container_{timestamp_str} {{ background-color: #f4f4f4; padding: 20px; overflow-y: auto; height: 100vh; }}
                :-moz-full-screen #fs_container_{timestamp_str} {{ background-color: #f4f4f4; padding: 20px; overflow-y: auto; height: 100vh; }}
                :-ms-fullscreen #fs_container_{timestamp_str} {{ background-color: #f4f4f4; padding: 20px; overflow-y: auto; height: 100vh; }}
            </style>

            <div id="fs_container_{timestamp_str}" style="width: 100%; background: #fff; padding: 10px; border-radius: 8px; box-sizing: border-box;">
                <div style="display: flex; justify-content: flex-end; margin-bottom: 5px; gap: 8px;">
                    <button id="btn_fullscreen_{timestamp_str}" style="cursor:pointer; padding:6px 12px; font-weight:bold; background:#343a40; color:white; border:none; border-radius:4px;">⛶ Full Screen</button>
                    <button id="btn_exit_fullscreen_{timestamp_str}" style="cursor:pointer; padding:6px 12px; font-weight:bold; background:#dc3545; color:white; border:none; border-radius:4px; display:none;">🗗 Exit Full Screen</button>
                </div>

                <div id="dashboard_wrapper_{timestamp_str}" style="display: flex; flex-direction: column; width: 100%; height: 95vh; min-height: 1000px; box-sizing: border-box; font-family: sans-serif; gap: 10px;">
                    <div id="top_row_{timestamp_str}" style="display: flex; flex-direction: row; flex: 1 1 50%; min-height: 45vh; gap: 10px; align-items: stretch;">
                        <div id="left_column_resizable_{timestamp_str}" style="width: 50%; min-width: 25%; max-width: 75%; display: flex; flex-direction: column; resize: horizontal; overflow: hidden; border: 1px solid #ddd; border-radius: 4px; flex-shrink: 0; background: white; z-index: 1;">
                            <div id="map_container_{timestamp_str}" style="flex-grow: 1; position: relative; min-height: 200px; z-index: 1;"></div>
                            <div id="bounds_ctrl_master_{timestamp_str}" style="width: 100%; padding: 8px 12px; background: #f8f9fa; border-top: 1px solid #ddd; font-size: 13px; display: flex; align-items: center; box-sizing: border-box; flex-shrink: 0; z-index: 2; gap: 10px;">
                                <div style="font-weight:bold; color:#333; flex-shrink:0;">Map Center:</div>
                                <div id="bounds_text_master_{timestamp_str}" style="font-family: monospace; color:#0056b3; font-weight:bold; font-size: 12.5px; flex-grow: 1; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; text-align: center;">Locating...</div>
                                <div style="display: flex; gap: 4px; flex-shrink:0;">
                                    <button id="pan_up_{timestamp_str}" title="Pan North" style="cursor:pointer; padding:2px 8px; background:#e9ecef; border:1px solid #ccc; border-radius:3px; font-weight:bold;">&#8593;</button>
                                    <button id="pan_down_{timestamp_str}" title="Pan South" style="cursor:pointer; padding:2px 8px; background:#e9ecef; border:1px solid #ccc; border-radius:3px; font-weight:bold;">&#8595;</button>
                                    <button id="pan_left_{timestamp_str}" title="Pan West" style="cursor:pointer; padding:2px 8px; background:#e9ecef; border:1px solid #ccc; border-radius:3px; font-weight:bold;">&#8592;</button>
                                    <button id="pan_right_{timestamp_str}" title="Pan East" style="cursor:pointer; padding:2px 8px; background:#e9ecef; border:1px solid #ccc; border-radius:3px; font-weight:bold;">&#8594;</button>
                                </div>
                            </div>
                            <div id="legends_container_master_{timestamp_str}" style="width: 100%; padding: 12px; background: #fff; border-top: 1px solid #ddd; box-sizing: border-box; flex-shrink: 0; overflow-y: auto; max-height: 120px; z-index: 2;">
                                <div style="font-weight:bold; color:#333; margin-bottom:8px; text-align:center; font-size:13px;">Map Color Keys</div>
                                {legends_html}
                            </div>
                        </div>
                        <div id="plot_flex_container_{timestamp_str}" style="display: flex; flex-direction: row; flex: 1 1 auto; min-width: 0; background: #fff; border: 1px solid #ddd; border-radius: 4px; overflow: hidden; box-sizing: border-box;">
                            <div id="{plot_div_id}" style="flex: 1 1 auto; min-width: 0; height: 100%; position: relative;"></div>
                            <div id="plot_legend_sidebar_{timestamp_str}" style="width: 320px; flex-shrink: 0; height: 100%; background: #f8f9fa; border-left: 1px solid #eee; padding: 0; box-sizing: border-box; display: flex; flex-direction: column; overflow-y: hidden;">
                                <div style="padding: 8px; display: flex; flex-direction: column; flex-grow: 1; overflow-y: auto;">
                                    <div style="font-weight:bold; color:#333; text-align:center; margin-bottom:6px; font-size:12px; border-bottom:1px solid #ddd; padding-bottom:4px;">
                                        Plot Traces <br> <span style="font-size: 10px; font-weight: normal; color: #666;">(Offset &#8593;&#8595; | Scale &#43;&#8722;)</span>
                                    </div>
                                    <div id="plot_trace_cbs_{timestamp_str}" style="display: flex; flex-direction: column; gap: 6px; overflow-y: auto; margin-bottom: 10px;"></div>

                                    <div style="margin-top: auto; padding-top: 10px; border-top: 1px solid #ccc; display: flex; flex-direction: column; gap: 6px; flex-shrink:0;">
                                        <div style="font-size:11px; font-weight:bold; color:#333;">Trace Presets:</div>
                                        <select id="trace_preset_select_{timestamp_str}" style="width:100%; font-size:11px; padding:2px;"></select>
                                        <div style="display:flex; gap:4px;">
                                            <button id="btn_load_trace_{timestamp_str}" style="flex:1; cursor:pointer; font-size:11px; background:#e9ecef; border:1px solid #ccc; border-radius:3px;">Load</button>
                                            <button id="btn_def_trace_{timestamp_str}" style="flex:1; cursor:pointer; font-size:11px; background:#ffc107; border:1px solid #ccc; border-radius:3px;">⭐ Def</button>
                                        </div>
                                        <input type="text" id="trace_preset_name_{timestamp_str}" placeholder="New name..." style="width:100%; font-size:11px; padding:2px; box-sizing:border-box;">
                                        <div style="display:flex; gap:4px; margin-top:2px;">
                                            <button id="btn_auto_stack_{timestamp_str}" style="flex:1; cursor:pointer; padding:4px; font-weight:bold; font-size:11px; background:#17a2b8; color:white; border:none; border-radius:3px;">🪄 Auto Stack</button>
                                            <button id="btn_save_trace_{timestamp_str}" style="flex:1; cursor:pointer; padding:4px; font-weight:bold; font-size:11px; background:#28a745; color:white; border:none; border-radius:3px;">💾 Save Trace</button>
                                        </div>
                                    </div>
                                </div>
                                <!-- Covariance Matrix Appended Here -->
                            </div>
                        </div>
                    </div>
                    <div id="bottom_row_{timestamp_str}" style="display: {'flex' if has_splom else 'none'}; flex-direction: row; flex: 1 1 50%; min-height: 45vh; width: 100%; gap: 10px; align-items: flex-start; overflow-x: auto; padding-bottom: 10px;">
                        <div id="splom_container_{timestamp_str}" style="height: 100%; aspect-ratio: 1; max-height: 1080px; position: relative; background: white; border: 1px solid #ddd; border-radius: 4px; box-sizing: border-box; display: flex; flex-direction: row; flex-shrink: 0;">
                            <div id="{splom_div_id}" style="flex-grow: 1; min-width: 0; height: 100%;"></div>
                            <div style="width: 35px; height: 80%; margin: auto 10px auto 0; position: relative; display: flex; flex-direction: column; align-items: center; justify-content: center;">
                                <div style="font-size: 10px; font-weight: bold; margin-bottom: 5px; color: #555; writing-mode: vertical-rl; transform: rotate(180deg);">Time Scrub</div>
                                <div id="time_scrub_bar_{timestamp_str}" style="width: 15px; height: 100%; background: linear-gradient(to top, #fde725, #35b779, #31688e, #440154); border: 1px solid #ccc; cursor: ns-resize; border-radius: 3px;"></div>
                            </div>
                        </div>
                    </div>
                </div>
            </div>

            <div id="custom_splom_tooltip_{timestamp_str}" style="position: absolute; display: none; background: rgba(0,0,0,0.85); color: white; padding: 10px; border-radius: 5px; font-size: 12px; font-family: sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.3); z-index: 99999; pointer-events: none;"></div>

            <script>
            (function() {{
                var telemetryData = {js_data if has_plots else '[]'};
                var mapId = '{map_id}';
                var plotDivId = '{plot_div_id}';
                var splomDivId = '{splom_div_id}';
                var covDivId = '{cov_div_id}';
                var hasPlots = {'true' if has_plots else 'false'};
                var hasSplom = {'true' if has_splom else 'false'};

                var dataPresetName = "{data_preset_name}";
                var tracePresets = {trace_presets_json};
                var activeTracePreset = "{global_startup_trace_preset}";

                var splomPlot = null;
                var myPlot = null;
                var hoverMarker = null;
                var isSyncing = false;
                var currentHoverIdx = -1;

                var validPlotCols = {valid_plot_cols_json};
                var displayLabelsMap = {display_labels_map_json};

                var customSplomTooltip = document.getElementById('custom_splom_tooltip_{timestamp_str}');

                document.addEventListener('mousemove', function(e) {{
                    if (customSplomTooltip.style.display === 'block') {{
                        customSplomTooltip.style.left = (e.clientX + 15) + 'px';
                        customSplomTooltip.style.top = (e.clientY + 15) + 'px';
                    }}
                }});

                function mountDashboardLayout() {{
                    var mapDiv = document.getElementById(mapId);
                    if (!mapDiv) {{
                        setTimeout(mountDashboardLayout, 50);
                        return;
                    }}

                    var mapContainer = document.getElementById('map_container_{timestamp_str}');
                    mapContainer.appendChild(mapDiv);
                    mapDiv.style.width = '100%';
                    mapDiv.style.height = '100%';
                    mapDiv.style.position = 'absolute';

                    var fsBtn = document.getElementById('btn_fullscreen_{timestamp_str}');
                    var exitFsBtn = document.getElementById('btn_exit_fullscreen_{timestamp_str}');
                    var fsContainer = document.getElementById('fs_container_{timestamp_str}');
                    var dashWrapper = document.getElementById('dashboard_wrapper_{timestamp_str}');

                    fsBtn.addEventListener('click', function() {{
                        var elem = fsContainer;
                        if (elem.requestFullscreen) {{
                            elem.requestFullscreen().catch(function(e) {{ console.log(e); }});
                        }} else if (elem.webkitRequestFullscreen) {{
                            elem.webkitRequestFullscreen();
                        }} else if (elem.mozRequestFullScreen) {{
                            elem.mozRequestFullScreen();
                        }} else if (elem.msRequestFullscreen) {{
                            elem.msRequestFullscreen();
                        }}
                    }});

                    exitFsBtn.addEventListener('click', function() {{
                        if (document.exitFullscreen) {{
                            document.exitFullscreen();
                        }} else if (document.webkitExitFullscreen) {{
                            document.webkitExitFullscreen();
                        }} else if (document.mozCancelFullScreen) {{
                            document.mozCancelFullScreen();
                        }} else if (document.msExitFullscreen) {{
                            document.msExitFullscreen();
                        }}
                    }});

                    function handleFSChange() {{
                        var isFS = document.fullscreenElement || document.webkitFullscreenElement || document.mozFullScreenElement || document.msFullscreenElement;
                        if (isFS) {{
                            fsContainer.style.overflowY = 'auto';
                            fsContainer.style.height = '100vh';
                            dashWrapper.style.height = 'calc(100vh - 40px)';
                            fsBtn.style.display = 'none';
                            exitFsBtn.style.display = 'block';
                        }} else {{
                            fsContainer.style.height = 'auto';
                            dashWrapper.style.height = '95vh';
                            fsBtn.style.display = 'block';
                            exitFsBtn.style.display = 'none';
                        }}
                        window.dispatchEvent(new Event('resize'));

                        if (window[mapId]) {{
                            setTimeout(function() {{ window[mapId].invalidateSize(); }}, 300);
                        }}
                    }}

                    document.addEventListener('fullscreenchange', handleFSChange);
                    document.addEventListener('webkitfullscreenchange', handleFSChange);
                    document.addEventListener('mozfullscreenchange', handleFSChange);
                    document.addEventListener('MSFullscreenChange', handleFSChange);

                    if (hasSplom) {{
                        var plotLegendDiv = document.getElementById('plot_legend_sidebar_{timestamp_str}');
                        var covWrapper = document.createElement('div');
                        covWrapper.style.cssText = 'width: 100%; border-top: 1px solid #ddd; background: #f8f9fa; display: flex; justify-content: center; align-items: center; padding: 15px 0; flex-shrink: 0;';

                        var covContainer = document.createElement('div');
                        covContainer.id = covDivId;
                        covContainer.style.cssText = 'width: 50%; aspect-ratio: 1; background: white; border: 1px solid #eee; box-sizing: border-box; box-shadow: 0 1px 3px rgba(0,0,0,0.1);';

                        covWrapper.appendChild(covContainer);
                        plotLegendDiv.appendChild(covWrapper);
                    }}

                    if (hasPlots) {{
                        myPlot = document.getElementById(plotDivId);

                        var features = {feats_json};
                        var featureLabels = {labels_json};
                        var colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'];

                        var numFeats = features.length;
                        var leftAxes = Math.ceil(numFeats / 2);
                        var rightAxes = Math.floor(numFeats / 2);
                        var offsetStep = 0.06;
                        var domainStart = (leftAxes > 1) ? (leftAxes - 1) * offsetStep : 0;
                        var domainEnd = (rightAxes > 1) ? 1.0 - (rightAxes - 1) * offsetStep : 1.0;
                        var layout = {{
                            title: 'Telemetry Dashboard',
                            hovermode: 'x',
                            showlegend: false,
                            autosize: true,
                            xaxis: {{
                                title: 'Time ({tz_abbr})',
                                domain: [domainStart, domainEnd],
                                showgrid: true,
                                showspikes: true,
                                spikemode: 'across',
                                spikedash: 'dash',
                                spikecolor: 'black',
                                spikethickness: 2
                            }},
                            margin: {{l: 40, r: 40, t: 40, b: 40}}
                        }};

                        var traces = [];
                        var cbElements = [];

                        var presetData = tracePresets[activeTracePreset] || {{}};

                        for (var i=0; i<numFeats; i++) {{
                            var isLeft = (i % 2 === 0);
                            var axisIndex = Math.floor(i / 2);
                            var yAxisName = i === 0 ? 'y' : 'y' + (i + 1);
                            var isLocalTime = featureLabels[i].indexOf("Local Time") !== -1;

                            var isVisible = true;
                            if (presetData[features[i]] && presetData[features[i]].visible !== undefined) {{
                                isVisible = presetData[features[i]].visible;
                            }}

                            var trace = {{
                                x: telemetryData.map(d => d.time_iso),
                                y: telemetryData.map(d => d[features[i]]),
                                name: featureLabels[i],
                                yaxis: yAxisName,
                                visible: isVisible
                            }};

                            if (isLocalTime) {{
                                trace.mode = 'markers';
                                trace.marker = {{
                                    color: telemetryData.map(d => d[features[i]]),
                                    colorscale: 'Viridis',
                                    size: 4
                                }};
                            }} else {{
                                trace.mode = 'lines';
                                trace.line = {{color: colors[i % colors.length], width: 1.5}};
                            }}
                            traces.push(trace);

                            var axisObj = {{
                                title: {{text: featureLabels[i], font: {{color: colors[i % colors.length], size: 10}}}},
                                tickfont: {{color: colors[i % colors.length], size: 9}},
                                showgrid: (i === 0),
                                zeroline: (i === 0),
                                side: isLeft ? 'left' : 'right',
                                visible: isVisible
                            }};

                            if (presetData[features[i]] && presetData[features[i]].span) {{
                                var center = presetData[features[i]].center;
                                var span = presetData[features[i]].span;
                                axisObj.range = [center - span/2, center + span/2];
                                axisObj.autorange = false;
                            }} else {{
                                var validData = telemetryData.map(d => d[features[i]]).filter(v => v !== null && !isNaN(v));
                                var dMin = Math.min(...validData);
                                var dMax = Math.max(...validData);
                                if (!isFinite(dMin)) {{ dMin = 0; dMax = 1; }}
                                if (dMin === dMax) {{ dMin -= 1; dMax += 1; }}
                                var center = (dMax + dMin) / 2.0;
                                var span = (dMax - dMin) * 1.1;
                                axisObj.range = [center - span/2, center + span/2];
                                axisObj.autorange = false;
                            }}

                            if (i !== 0) {{ axisObj.overlaying = 'y'; }}

                            if (axisIndex > 0) {{
                                axisObj.anchor = 'free';
                                axisObj.position = isLeft ? (domainStart - axisIndex * offsetStep) : (domainEnd + axisIndex * offsetStep);
                            }} else {{
                                axisObj.anchor = 'x';
                            }}

                            if (i === 0) {{ layout.yaxis = axisObj; }}
                            else {{ layout['yaxis' + (i + 1)] = axisObj; }}
                        }}

                        var cbContainer = document.getElementById('plot_trace_cbs_{timestamp_str}');

                        for (var i=0; i<numFeats; i++) {{
                            (function(idx) {{
                                var featKey = features[idx];
                                var color = featureLabels[idx].indexOf("Local Time") !== -1 ? '#000' : colors[idx % colors.length];

                                var rowDiv = document.createElement('div');
                                rowDiv.style.cssText = 'display: flex; justify-content: space-between; align-items: center; width: 100%;';

                                var lbl = document.createElement('label');
                                lbl.style.cssText = 'display: flex; align-items: center; cursor: pointer; font-weight: bold; color: ' + color + '; font-size: 12px; margin: 0; line-height: 1.2; flex-grow: 1; overflow: hidden;';

                                var cb = document.createElement('input');
                                cb.type = 'checkbox';
                                cb.checked = traces[idx].visible;
                                cb.style.marginRight = '5px';
                                cb.style.flexShrink = '0';
                                cbElements[idx] = cb;

                                cb.addEventListener('change', function(e) {{
                                    var isChecked = e.target.checked;
                                    var update = {{ visible: isChecked }};
                                    var layoutUpdate = {{}};
                                    var axisKey = idx === 0 ? 'yaxis' : 'yaxis' + (idx + 1);
                                    layoutUpdate[axisKey + '.visible'] = isChecked;
                                    Plotly.update(myPlot, update, layoutUpdate, [idx]);
                                }});

                                var textNode = document.createElement('span');
                                textNode.innerText = featureLabels[idx];
                                textNode.title = featureLabels[idx];
                                textNode.style.whiteSpace = 'nowrap';
                                textNode.style.overflow = 'hidden';
                                textNode.style.textOverflow = 'ellipsis';

                                lbl.appendChild(cb);
                                lbl.appendChild(textNode);

                                var btnContainer = document.createElement('div');
                                btnContainer.style.cssText = 'display: flex; gap: 4px; margin-left: auto; flex-shrink: 0;';

                                var btnCol1 = document.createElement('div');
                                btnCol1.style.cssText = 'display: flex; flex-direction: row; gap: 2px; align-items: center;';

                                var btnShiftUp = document.createElement('button');
                                btnShiftUp.innerHTML = '&#8593;';
                                btnShiftUp.title = 'Shift Trace Up (Lower Axis Center)';
                                btnShiftUp.style.cssText = 'cursor:pointer; font-size:11px; font-weight:bold; padding:2px 4px; line-height:1; border:1px solid #ccc; background:#e9ecef; border-radius:2px; color:#333;';

                                var btnShiftDown = document.createElement('button');
                                btnShiftDown.innerHTML = '&#8595;';
                                btnShiftDown.title = 'Shift Trace Down (Raise Axis Center)';
                                btnShiftDown.style.cssText = 'cursor:pointer; font-size:11px; font-weight:bold; padding:2px 4px; line-height:1; border:1px solid #ccc; background:#e9ecef; border-radius:2px; color:#333;';

                                var btnCol2 = document.createElement('div');
                                btnCol2.style.cssText = 'display: flex; flex-direction: row; gap: 2px; align-items: center; margin-left: 2px;';

                                var btnZoomIn = document.createElement('button');
                                btnZoomIn.innerHTML = '&#43;';
                                btnZoomIn.title = 'Expand Trace Scale (Zoom In)';
                                btnZoomIn.style.cssText = 'cursor:pointer; font-size:11px; font-weight:bold; padding:2px 4px; line-height:1; border:1px solid #ccc; background:#e9ecef; border-radius:2px; color:#333;';

                                var btnZoomOut = document.createElement('button');
                                btnZoomOut.innerHTML = '&#8722;';
                                btnZoomOut.title = 'Shrink Trace Scale (Zoom Out)';
                                btnZoomOut.style.cssText = 'cursor:pointer; font-size:11px; font-weight:bold; padding:2px 4px; line-height:1; border:1px solid #ccc; background:#e9ecef; border-radius:2px; color:#333;';

                                btnShiftUp.addEventListener('click', function(e) {{
                                    e.preventDefault(); e.stopPropagation();
                                    var axisKey = idx === 0 ? 'yaxis' : 'yaxis' + (idx + 1);
                                    var ax = myPlot._fullLayout[axisKey];
                                    if (!ax) return;
                                    var r0 = parseFloat(ax.range[0]);
                                    var r1 = parseFloat(ax.range[1]);
                                    var span = r1 - r0;
                                    var shift = span * 0.05;
                                    var layoutUpdate = {{}};
                                    layoutUpdate[axisKey + '.range'] = [r0 - shift, r1 - shift];
                                    layoutUpdate[axisKey + '.autorange'] = false;
                                    Plotly.relayout(myPlot, layoutUpdate);
                                }});

                                btnShiftDown.addEventListener('click', function(e) {{
                                    e.preventDefault(); e.stopPropagation();
                                    var axisKey = idx === 0 ? 'yaxis' : 'yaxis' + (idx + 1);
                                    var ax = myPlot._fullLayout[axisKey];
                                    if (!ax) return;
                                    var r0 = parseFloat(ax.range[0]);
                                    var r1 = parseFloat(ax.range[1]);
                                    var span = r1 - r0;
                                    var shift = span * 0.05;
                                    var layoutUpdate = {{}};
                                    layoutUpdate[axisKey + '.range'] = [r0 + shift, r1 + shift];
                                    layoutUpdate[axisKey + '.autorange'] = false;
                                    Plotly.relayout(myPlot, layoutUpdate);
                                }});

                                btnZoomIn.addEventListener('click', function(e) {{
                                    e.preventDefault(); e.stopPropagation();
                                    var axisKey = idx === 0 ? 'yaxis' : 'yaxis' + (idx + 1);
                                    var ax = myPlot._fullLayout[axisKey];
                                    if (!ax) return;
                                    var r0 = parseFloat(ax.range[0]);
                                    var r1 = parseFloat(ax.range[1]);
                                    var span = r1 - r0;
                                    var center = (r1 + r0) / 2.0;
                                    var newSpan = span / 1.25;
                                    var layoutUpdate = {{}};
                                    layoutUpdate[axisKey + '.range'] = [center - newSpan/2, center + newSpan/2];
                                    layoutUpdate[axisKey + '.autorange'] = false;
                                    Plotly.relayout(myPlot, layoutUpdate);
                                }});

                                btnZoomOut.addEventListener('click', function(e) {{
                                    e.preventDefault(); e.stopPropagation();
                                    var axisKey = idx === 0 ? 'yaxis' : 'yaxis' + (idx + 1);
                                    var ax = myPlot._fullLayout[axisKey];
                                    if (!ax) return;
                                    var r0 = parseFloat(ax.range[0]);
                                    var r1 = parseFloat(ax.range[1]);
                                    var span = r1 - r0;
                                    var center = (r1 + r0) / 2.0;
                                    var newSpan = span * 1.25;
                                    var layoutUpdate = {{}};
                                    layoutUpdate[axisKey + '.range'] = [center - newSpan/2, center + newSpan/2];
                                    layoutUpdate[axisKey + '.autorange'] = false;
                                    Plotly.relayout(myPlot, layoutUpdate);
                                }});

                                btnCol1.appendChild(btnShiftUp);
                                btnCol1.appendChild(btnShiftDown);
                                btnCol2.appendChild(btnZoomIn);
                                btnCol2.appendChild(btnZoomOut);

                                btnContainer.appendChild(btnCol1);
                                btnContainer.appendChild(btnCol2);

                                rowDiv.appendChild(lbl);
                                rowDiv.appendChild(btnContainer);
                                cbContainer.appendChild(rowDiv);
                            }})(i);
                        }}

                        document.getElementById('btn_auto_stack_{timestamp_str}').onclick = function() {{
                            var visibleIndices = [];
                            for(var i=0; i<numFeats; i++) {{
                                if(cbElements[i].checked) visibleIndices.push(i);
                            }}
                            var V = visibleIndices.length;
                            if(V === 0) return;

                            var layoutUpdate = {{}};
                            for(var v=0; v<V; v++) {{
                                var idx = visibleIndices[v];
                                var featKey = features[idx];
                                var axisKey = idx === 0 ? 'yaxis' : 'yaxis' + (idx + 1);

                                var validData = telemetryData.map(d => d[featKey]).filter(val => val !== null && !isNaN(val));
                                var dMin = Math.min(...validData);
                                var dMax = Math.max(...validData);
                                if(!isFinite(dMin)) {{ dMin = 0; dMax = 1; }}
                                if(dMin === dMax) {{ dMin -= 1; dMax += 1; }}

                                var S = dMax - dMin;
                                var Spad = S * 1.1;
                                var Dcenter = (dMax + dMin) / 2.0;

                                var Rspan = Spad * V;
                                var Rcenter = Dcenter + Rspan * ((v + 0.5 - V/2.0) / V);

                                layoutUpdate[axisKey + '.range'] = [Rcenter - Rspan/2, Rcenter + Rspan/2];
                                layoutUpdate[axisKey + '.autorange'] = false;
                            }}
                            Plotly.relayout(myPlot, layoutUpdate);
                        }};

                        var telemetryConfig = {{
                            responsive: true,
                            toImageButtonOptions: {{
                                format: 'png',
                                filename: dataPresetName + '_' + activeTracePreset + '_Telemetry_{timestamp_str}'
                            }}
                        }};

                        Plotly.newPlot(myPlot, traces, layout, telemetryConfig).then(function() {{
                            Plotly.Plots.resize(myPlot);
                        }});

                        var plotObserver = new ResizeObserver(function() {{
                            if (myPlot) Plotly.Plots.resize(myPlot);
                        }});
                        plotObserver.observe(document.getElementById('plot_flex_container_{timestamp_str}'));

                        // Trace Preset UI Logic
                        var presetSelect = document.getElementById('trace_preset_select_{timestamp_str}');
                        function populateTracePresets() {{
                            presetSelect.innerHTML = '';
                            for (var k in tracePresets) {{
                                var opt = document.createElement('option');
                                opt.value = k; opt.text = k;
                                if (k === activeTracePreset) opt.selected = true;
                                presetSelect.appendChild(opt);
                            }}
                        }}
                        populateTracePresets();

                        document.getElementById('btn_load_trace_{timestamp_str}').onclick = function() {{
                            var name = presetSelect.value;
                            if (!tracePresets[name]) return;
                            var states = tracePresets[name];
                            activeTracePreset = name;

                            var layoutUpdate = {{}};
                            var update = {{visible: []}};
                            var traceIndices = [];

                            for(var i=0; i<numFeats; i++) {{
                                var axKey = i === 0 ? 'yaxis' : 'yaxis' + (i+1);
                                traceIndices.push(i);
                                if (states[features[i]]) {{
                                    var s = states[features[i]];
                                    layoutUpdate[axKey + '.range'] = [s.center - s.span/2, s.center + s.span/2];
                                    layoutUpdate[axKey + '.autorange'] = false;
                                    layoutUpdate[axKey + '.visible'] = s.visible;
                                    update.visible.push(s.visible);
                                    cbElements[i].checked = s.visible;
                                }} else {{
                                    update.visible.push(traces[i].visible);
                                }}
                            }}

                            Plotly.update(myPlot, update, layoutUpdate, traceIndices);
                            myPlot._context.toImageButtonOptions.filename = dataPresetName + '_' + activeTracePreset + '_Telemetry_{timestamp_str}';
                        }};

                        document.getElementById('btn_def_trace_{timestamp_str}').onclick = function() {{
                            var name = presetSelect.value;
                            if (typeof google !== 'undefined' && google.colab && google.colab.kernel) {{
                                google.colab.kernel.invokeFunction('update_trace_presets', [JSON.stringify({{action: 'set_default', name: name}})], {{}});
                                var btn = document.getElementById('btn_def_trace_{timestamp_str}');
                                btn.innerHTML = '✅ Set!';
                                setTimeout(() => btn.innerHTML = '⭐ Def', 2000);
                            }}
                        }};

                        var btnSaveLayout = document.getElementById('btn_save_trace_{timestamp_str}');
                        btnSaveLayout.onclick = function() {{
                            var currentStates = {{}};
                            for(var i=0; i<numFeats; i++) {{
                                var axKey = i === 0 ? 'yaxis' : 'yaxis' + (i+1);
                                var ax = myPlot._fullLayout ? myPlot._fullLayout[axKey] : null;
                                var isVis = cbElements[i].checked;
                                if(ax && ax.range) {{
                                    var r0 = parseFloat(ax.range[0]);
                                    var r1 = parseFloat(ax.range[1]);
                                    var span = r1 - r0;
                                    var center = (r1 + r0) / 2.0;
                                    currentStates[features[i]] = {{center: center, span: span, visible: isVis}};
                                }}
                            }}

                            var inputName = document.getElementById('trace_preset_name_{timestamp_str}').value.trim();
                            var saveName = inputName ? inputName : presetSelect.value;

                            tracePresets[saveName] = currentStates;
                            activeTracePreset = saveName;
                            populateTracePresets();
                            document.getElementById('trace_preset_name_{timestamp_str}').value = '';

                            if (typeof google !== 'undefined' && google.colab && google.colab.kernel) {{
                                google.colab.kernel.invokeFunction('update_trace_presets', [JSON.stringify({{action: 'save', name: saveName, states: currentStates}})], {{}});
                            }}

                            myPlot._context.toImageButtonOptions.filename = dataPresetName + '_' + activeTracePreset + '_Telemetry_{timestamp_str}';

                            btnSaveLayout.innerHTML = '✅ Saved!';
                            setTimeout(() => btnSaveLayout.innerHTML = '💾 Save Trace', 2000);
                        }};
                    }}

                    if (hasSplom) {{
                        splomPlot = document.getElementById(splomDivId);
                        var covContainer = document.getElementById(covDivId);

                        var splomDims = [];
                        for (var i=0; i<validPlotCols.length; i++) {{
                            splomDims.push({{
                                label: displayLabelsMap[validPlotCols[i]],
                                values: telemetryData.map(d => d[validPlotCols[i]])
                            }});
                        }}

                        var splomData = [
                            {{
                                type: 'splom',
                                dimensions: splomDims,
                                text: telemetryData.map(d => 'Clock: ' + d.time_str_hover),
                                marker: {{
                                    color: telemetryData.map(d => d.elapsed_minutes),
                                    colorscale: 'Viridis',
                                    size: 4,
                                    showscale: false
                                }},
                                hoverinfo: 'none'
                            }}
                        ];

                        var splomLayout = {{
                            title: 'Interactive Scatter Matrix',
                            margin: {{l: 60, r: 20, t: 40, b: 60}},
                            hovermode: 'closest',
                            dragmode: 'pan',
                            width: 1080,
                            height: 1080
                        }};

                        for (var i=0; i<validPlotCols.length; i++) {{
                            var xKey = i === 0 ? 'xaxis' : 'xaxis' + (i+1);
                            var yKey = i === 0 ? 'yaxis' : 'yaxis' + (i+1);
                            splomLayout[xKey] = {{ title: {{ text: displayLabelsMap[validPlotCols[i]], font: {{size: 10}} }} }};
                            splomLayout[yKey] = {{ title: {{ text: displayLabelsMap[validPlotCols[i]], font: {{size: 10}} }} }};
                        }}

                        var splomConfig = {{
                            responsive: true,
                            toImageButtonOptions: {{ format: 'png', filename: dataPresetName + '_ScatterMatrix_{timestamp_str}' }}
                        }};

                        Plotly.newPlot(splomPlot, splomData, splomLayout, splomConfig).then(() => {{
                            splomPlot.on('plotly_hover', function(data) {{
                                if (isSyncing) return;
                                if (data && data.points && data.points.length > 0) {{
                                    var pt = data.points[0];
                                    var idx = pt.pointIndex;
                                    var ptX = pt.x;
                                    var ptY = pt.y;

                                    var row = telemetryData[idx];
                                    var xName = "X"; var yName = "Y";
                                    for (var i=0; i<validPlotCols.length; i++) {{
                                        var col = validPlotCols[i];
                                        if (Math.abs(row[col] - ptX) < 0.0001) xName = displayLabelsMap[col];
                                        if (Math.abs(row[col] - ptY) < 0.0001) yName = displayLabelsMap[col];
                                    }}

                                    var timeStr = row.time_str_hover;
                                    var xValStr = (typeof ptX === 'number') ? ptX.toFixed(2) : ptX;
                                    var yValStr = (typeof ptY === 'number') ? ptY.toFixed(2) : ptY;

                                    customSplomTooltip.innerHTML = '<b style="color:#fde725;">Clock: ' + timeStr + '</b><hr style="margin:4px 0;border-top:1px solid #555;">' +
                                                                   '<b>' + yName + ':</b> ' + yValStr + '<br>' +
                                                                   '<b>' + xName + ':</b> ' + xValStr;
                                    customSplomTooltip.style.display = 'block';

                                    window.requestAnimationFrame(function() {{
                                        syncAllViews(idx, 'splom');
                                    }});
                                }}
                            }});
                            splomPlot.on('plotly_unhover', function() {{
                                customSplomTooltip.style.display = 'none';
                                clearHover();
                            }});
                        }});

                        var covMatrix = {cov_matrix_json};
                        var corrMatrix = {corr_matrix_json};
                        var textMatrix = [];
                        for(var r=0; r<covMatrix.length; r++) {{
                            var rArr = [];
                            for(var c=0; c<covMatrix[r].length; c++) {{
                                rArr.push(corrMatrix[r][c].toFixed(3));
                            }}
                            textMatrix.push(rArr);
                        }}

                        var covData = [{{
                            type: 'heatmap',
                            z: covMatrix,
                            x: validPlotCols.map(c => displayLabelsMap[c]),
                            y: validPlotCols.map(c => displayLabelsMap[c]),
                            customdata: textMatrix,
                            hovertemplate: 'Feature 1: %{{y}}<br>Feature 2: %{{x}}<br>Covariance: %{{z:.2g}}<br>Correlation (r): %{{customdata}}<extra></extra>',
                            colorscale: 'RdBu',
                            zmid: 0
                        }}];

                        var covConfig = {{
                            responsive: true,
                            toImageButtonOptions: {{ format: 'png', filename: dataPresetName + '_CovarianceMatrix_{timestamp_str}' }}
                        }};

                        Plotly.newPlot(covContainer, covData, {{
                            title: {{text: 'Covariance Matrix', font: {{size: 13}} }},
                            margin: {{l: 20, r: 20, t: 40, b: 20}},
                            xaxis: {{ showticklabels: false, ticks: '' }},
                            yaxis: {{ showticklabels: false, ticks: '', scaleanchor: 'x', scaleratio: 1 }}
                        }}, covConfig);

                        var colorbarDiv = document.getElementById('time_scrub_bar_{timestamp_str}');
                        if (colorbarDiv) {{
                            colorbarDiv.onmousemove = function(e) {{
                                var rect = colorbarDiv.getBoundingClientRect();
                                var y = e.clientY - rect.top;
                                var frac = 1 - (y / rect.height);
                                frac = Math.max(0, Math.min(1, frac));

                                var elapsed = telemetryData.map(d => d.elapsed_minutes);
                                var minTime = Math.min(...elapsed);
                                var maxTime = Math.max(...elapsed);
                                var targetTime = minTime + frac * (maxTime - minTime);

                                var minDist = Infinity;
                                var closestIdx = 0;
                                for (var i=0; i<telemetryData.length; i++) {{
                                    var d = Math.abs(telemetryData[i].elapsed_minutes - targetTime);
                                    if (d < minDist) {{ minDist = d; closestIdx = i; }}
                                }}
                                window.requestAnimationFrame(function() {{
                                    syncAllViews(closestIdx, 'scrub');
                                }});
                            }};
                            colorbarDiv.onmouseout = clearHover;
                        }}
                    }}

                    wireMapSync();
                }}

                var firstLayer = "{first_layer_name}";
                var mapFeatureNames = {map_feature_names_json};
                var layerJsMap = {layer_js_map_json};
                var legendJsMap = {legend_js_map_json};

                function updateLegends(activeLayerName) {{
                    var activeLegendId = legendJsMap[activeLayerName];
                    Object.keys(legendJsMap).forEach(function(key) {{
                        var legId = legendJsMap[key];
                        var el = document.getElementById(legId);
                        if (el) {{ el.style.display = (legId === activeLegendId) ? 'block' : 'none'; }}
                    }});
                    var container = document.getElementById('legends_container_master_{timestamp_str}');
                    if (container) container.style.display = activeLegendId ? 'block' : 'none';
                }}

                function syncAllViews(idx, source) {{
                    var shouldShowTooltip = (source === 'map_direct');

                    if (idx === currentHoverIdx) {{
                        if (hoverMarker) {{
                            if (shouldShowTooltip && !hoverMarker.isTooltipOpen()) {{
                                hoverMarker.setTooltipContent(telemetryData[idx].tooltip_html);
                                hoverMarker.openTooltip();
                            }} else if (!shouldShowTooltip && hoverMarker.isTooltipOpen()) {{
                                hoverMarker.closeTooltip();
                            }}
                        }}
                        return;
                    }}

                    currentHoverIdx = idx;
                    isSyncing = true;

                    try {{
                        if (hoverMarker) {{
                            hoverMarker.setLatLng([telemetryData[idx].lat, telemetryData[idx].lon]);
                            if (shouldShowTooltip) {{
                                hoverMarker.setTooltipContent(telemetryData[idx].tooltip_html);
                                if (!hoverMarker.isTooltipOpen()) hoverMarker.openTooltip();
                            }} else {{
                                if (hoverMarker.isTooltipOpen()) hoverMarker.closeTooltip();
                            }}
                        }}

                        if (myPlot && source !== 'plot') {{
                            var layoutUpdate = {{
                                shapes: [{{
                                    type: 'line',
                                    x0: telemetryData[idx].time_iso,
                                    x1: telemetryData[idx].time_iso,
                                    y0: 0,
                                    y1: 1,
                                    yref: 'paper',
                                    xref: 'x',
                                    line: {{ color: 'black', width: 2, dash: 'dash' }},
                                    layer: 'above'
                                }}]
                            }};
                            Plotly.relayout(myPlot, layoutUpdate).catch(e => console.error(e));
                        }}

                        if (splomPlot && hasSplom) {{
                            Plotly.restyle(splomPlot, 'selectedpoints', [[idx]], [0]).catch(e => console.error(e));
                        }}
                    }} catch (e) {{
                        console.error("Sync error:", e);
                    }} finally {{
                        isSyncing = false;
                    }}
                }}

                function clearHover() {{
                    if (isSyncing) return;
                    currentHoverIdx = -1;

                    try {{
                        if (hoverMarker && hoverMarker.isTooltipOpen()) hoverMarker.closeTooltip();

                        if (myPlot) {{
                            Plotly.Fx.unhover(myPlot);
                            Plotly.relayout(myPlot, {{shapes: []}}).catch(e => console.error(e));
                        }}

                        if (splomPlot && hasSplom) {{
                            Plotly.restyle(splomPlot, 'selectedpoints', [null], [0]).catch(e => console.error(e));
                        }}
                    }} catch (e) {{
                        console.error("Clear hover error:", e);
                    }}
                }}

                function wireMapSync() {{
                    var initMapSync = setInterval(function() {{
                        var myMap = window[mapId];

                        if (myMap) {{
                            clearInterval(initMapSync);

                            var featureLayers = {{}};
                            mapFeatureNames.forEach(function(name) {{
                                var layerVar = layerJsMap[name];
                                if (window[layerVar]) {{ featureLayers[name] = window[layerVar]; }}
                            }});

                            var DropdownControl = L.Control.extend({{
                                options: {{ position: 'topright' }},
                                onAdd: function (map) {{
                                    var div = L.DomUtil.create('div', 'custom-dropdown');
                                    div.style.backgroundColor = 'white';
                                    div.style.padding = '5px 8px';
                                    div.style.border = '2px solid rgba(0,0,0,0.2)';
                                    div.style.borderRadius = '4px';
                                    div.style.boxShadow = '0 1px 5px rgba(0,0,0,0.4)';

                                    var label = document.createElement('span');
                                    label.innerHTML = '<b>Layer: </b>';
                                    label.style.marginRight = '5px';
                                    div.appendChild(label);
                                    var select = document.createElement('select');
                                    select.id = 'map_layer_select';
                                    select.style.fontSize = '14px';
                                    select.style.padding = '2px 5px';
                                    select.style.cursor = 'pointer';

                                    mapFeatureNames.forEach(function(name) {{
                                        if (featureLayers[name]) {{
                                            var option = document.createElement('option');
                                            option.value = name;
                                            option.text = name;
                                            if(name === firstLayer) {{ option.selected = true; }}
                                            select.appendChild(option);
                                        }}
                                    }});

                                    L.DomEvent.disableClickPropagation(div);
                                    L.DomEvent.disableScrollPropagation(div);

                                    select.addEventListener('change', function(e) {{
                                        var selectedName = e.target.value;
                                        mapFeatureNames.forEach(function(name) {{
                                            if (featureLayers[name] && myMap.hasLayer(featureLayers[name])) {{
                                                myMap.removeLayer(featureLayers[name]);
                                            }}
                                        }});
                                        if (featureLayers[selectedName]) {{
                                            myMap.addLayer(featureLayers[selectedName]);
                                        }}
                                        updateLegends(selectedName);
                                    }});

                                    div.appendChild(select);
                                    return div;
                                }}
                            }});
                            myMap.addControl(new DropdownControl());

                            mapFeatureNames.forEach(function(name) {{
                                if (name !== firstLayer && featureLayers[name] && myMap.hasLayer(featureLayers[name])) {{
                                    myMap.removeLayer(featureLayers[name]);
                                }}
                            }});
                            setTimeout(function() {{ updateLegends(firstLayer); }}, 500);

                            var leftCol = document.getElementById('left_column_resizable_{timestamp_str}');
                            if (leftCol) {{
                                var mapObserver = new ResizeObserver(function() {{ myMap.invalidateSize(); }});
                                mapObserver.observe(leftCol);
                            }}

                            hoverMarker = L.circleMarker([0, 0], {{
                                color: 'black', fillColor: 'white', fillOpacity: 1, radius: 7, weight: 2, interactive: false
                            }}).addTo(myMap);

                            var tooltip = L.tooltip({{direction: 'right', offset: [10, 0], opacity: 0.95}});
                            hoverMarker.bindTooltip(tooltip);

                            var textBounds = document.getElementById('bounds_text_master_{timestamp_str}');
                            function updateBoundsText() {{
                                var c = myMap.getCenter();
                                var latStr = Math.abs(c.lat).toFixed(4) + (c.lat >= 0 ? '° N' : '° S');
                                var lonStr = Math.abs(c.lng).toFixed(4) + (c.lng >= 0 ? '° E' : '° W');
                                if(textBounds) textBounds.innerHTML = latStr + ',  ' + lonStr;
                            }}
                            myMap.on('moveend', updateBoundsText);
                            setTimeout(updateBoundsText, 500);

                            var p_up = document.getElementById('pan_up_{timestamp_str}');
                            var p_dn = document.getElementById('pan_down_{timestamp_str}');
                            var p_lt = document.getElementById('pan_left_{timestamp_str}');
                            var p_rt = document.getElementById('pan_right_{timestamp_str}');

                            if(p_up) p_up.onclick = () => myMap.panBy([0, -150]);
                            if(p_dn) p_dn.onclick = () => myMap.panBy([0, 150]);
                            if(p_lt) p_lt.onclick = () => myMap.panBy([-150, 0]);
                            if(p_rt) p_rt.onclick = () => myMap.panBy([150, 0]);

                            myMap.on('mousemove', function(e) {{
                                if (isSyncing || typeof telemetryData === 'undefined' || telemetryData.length === 0) return;
                                var lat = e.latlng.lat;
                                var lon = e.latlng.lng;
                                var minDist = Infinity;
                                var minIdx = -1;
                                var cosLat = Math.cos(lat * Math.PI / 180.0);

                                for(var i=0; i<telemetryData.length; i++) {{
                                    var dlat = telemetryData[i].lat - lat;
                                    var dlon = (telemetryData[i].lon - lon) * cosLat;
                                    var dist = dlat*dlat + dlon*dlon;
                                    if(dist < minDist) {{ minDist = dist; minIdx = i; }}
                                }}

                                if (minIdx !== -1) {{
                                    var isDirectHover = minDist < 0.000001;
                                    window.requestAnimationFrame(function() {{
                                        syncAllViews(minIdx, isDirectHover ? 'map_direct' : 'map_remote');
                                    }});
                                }} else {{
                                    clearHover();
                                }}
                            }});

                            myMap.on('mouseout', clearHover);

                            myMap.on('click', function(e) {{
                                if (currentHoverIdx !== -1 && typeof telemetryData !== 'undefined' && telemetryData.length > 0) {{
                                    var textToCopy = telemetryData[currentHoverIdx].tooltip_text;
                                    var textArea = document.createElement("textarea");
                                    textArea.value = textToCopy;
                                    textArea.style.position = "fixed";
                                    textArea.style.top = "0";
                                    textArea.style.left = "0";
                                    document.body.appendChild(textArea);
                                    textArea.focus();
                                    textArea.select();
                                    try {{
                                        if (document.execCommand('copy')) {{
                                            var origHtml = telemetryData[currentHoverIdx].tooltip_html;
                                            hoverMarker.setTooltipContent("<div style='color:#0d7a22; font-weight:bold; margin-bottom:5px; border-bottom:1px solid #ccc; padding-bottom:3px;'>✅ Copied to clipboard!</div>" + origHtml);
                                            if (!hoverMarker.isTooltipOpen()) hoverMarker.openTooltip();
                                            setTimeout(function() {{
                                                if (currentHoverIdx !== -1) hoverMarker.setTooltipContent(telemetryData[currentHoverIdx].tooltip_html);
                                            }}, 1500);
                                        }}
                                    }} catch (err) {{}}
                                    document.body.removeChild(textArea);
                                }}
                            }});
                        }}

                        if (myPlot && typeof telemetryData !== 'undefined' && telemetryData.length > 0) {{
                            myPlot.on('plotly_hover', function(data){{
                                if (isSyncing) return;
                                var pt = data.points[0].pointIndex;
                                window.requestAnimationFrame(function() {{
                                    syncAllViews(pt, 'plot');
                                }});
                            }});
                            myPlot.on('plotly_unhover', clearHover);
                        }}
                    }}, 200);
                }}

                mountDashboardLayout();
            }})();
            </script>
            """

            f_feat.get_root().html.add_child(folium.Element(custom_html))

            # Inject allowfullscreen to bypass Colab iframe sandboxing
            html_str = f_feat._repr_html_()
            html_str = html_str.replace('<iframe ', '<iframe allowfullscreen="true" webkitallowfullscreen="true" mozallowfullscreen="true" msallowfullscreen="true" ')
            display(HTML(html_str))

            try:
                if os.path.exists(SETTINGS_FILE):
                    with open(SETTINGS_FILE, 'r') as f:
                        cur_settings = json.load(f)
                else: cur_settings = {}
                cur_settings['timing_ema'] = global_timing
                with open(SETTINGS_FILE, 'w') as f: json.dump(cur_settings, f)
            except: pass

        progress_html.value = ""

# 4. Wire up buttons and initialize
btn_up.on_click(on_up_clicked)
btn_refresh.on_click(on_scan_clicked)
btn_load.on_click(process_data)
btn_plot.on_click(generate_visualizations)
dir_select.observe(on_dir_change, names='value')

ui_container.children = [
    path_label,
    widgets.HBox([dir_select, widgets.VBox([btn_up, btn_refresh])]),
    widgets.VBox([gpx_dropdown, hr_dropdown, rr_dropdown, summary_dropdown]),
    btn_load,
    feature_box,
    btn_plot
]

# Display layout
display(widgets.HBox([btn_toggle_ui, progress_html]))
display(ui_container)
display(out)

# Initialize the first view and trigger the auto-scan
update_browser(start_path)

Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Output()